In [1]:
# Insurance Analytics Project

### Notebook 03: Feature Engineering & Gold Layer Creation

## Objective:

# Create business features
# Create analytical dimensions
# Build KPI-ready datasets
# Create Gold Layer datasets
# Prepare for SQL and BI dashboards

# Author: Chetan Sakate

In [2]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

In [3]:
customer_df = pd.read_csv(
    "../data/processed/cleaned_customer.csv"
)

policy_df = pd.read_csv(
    "../data/processed/cleaned_policy.csv"
)

claims_df = pd.read_csv(
    "../data/processed/cleaned_claims.csv"
)

payment_df = pd.read_csv(
    "../data/processed/cleaned_payments.csv"
)

additional_df = pd.read_csv(
    "../data/processed/cleaned_additional.csv"
)

In [4]:
policy_df["Policy Start Date"] = pd.to_datetime(
    policy_df["Policy Start Date"]
)

policy_df["Policy End Date"] = pd.to_datetime(
    policy_df["Policy End Date"]
)

claims_df["Date of Claim"] = pd.to_datetime(
    claims_df["Date of Claim"]
)

claims_df["Settlement Date"] = pd.to_datetime(
    claims_df["Settlement Date"]
)

payment_df["Date of Payment"] = pd.to_datetime(
    payment_df["Date of Payment"]
)

In [5]:
# FEATURE ENGINEERING

customer_df["Age Bucket"].value_counts()

Age Bucket
56+      2162
26-35     788
46-55     740
36-45     739
18-25     571
Name: count, dtype: int64

In [6]:
customer_df["Married Flag"] = np.where(
    customer_df["Marital Status"]=="Married",
    1,
    0
)

In [7]:
customer_df["Senior Citizen Flag"] = np.where(
    customer_df["Age"] >= 60,
    1,
    0
)

In [8]:
def customer_segment(age):

    if age <= 30:
        return "Young Adult"

    elif age <= 50:
        return "Middle Age"

    else:
        return "Senior Adult"

In [9]:
customer_df["Customer Segment"] = (
    customer_df["Age"]
    .apply(customer_segment)
)

In [10]:
# Policy Intelligence

policy_df["Policy Duration Days"] = (
    policy_df["Policy End Date"]
    -
    policy_df["Policy Start Date"]
).dt.days

In [11]:
def duration_bucket(days):

    if days <= 365:
        return "Short Term"

    elif days <= 1095:
        return "Medium Term"

    else:
        return "Long Term"

In [12]:
policy_df["Policy Duration Category"] = (
    policy_df["Policy Duration Days"]
    .apply(duration_bucket)
)

In [13]:
policy_df["Active Policy Flag"] = np.where(
    policy_df["Status"]=="Active",
    1,
    0
)

In [14]:
# Claims Intelligence

claims_df["Settlement Missing Flag"] = np.where(
    claims_df["Settlement Date"].isnull(),
    1,
    0
)

In [15]:
def claim_category(x):

    if x <= 5000:
        return "Low Claim"

    elif x <= 20000:
        return "Medium Claim"

    else:
        return "High Claim"

In [16]:
claims_df["Claim Amount Category"] = (
    claims_df["Claim Amount"]
    .apply(claim_category)
)

In [17]:
claims_df["Settlement Days"] = (
    claims_df["Settlement Date"]
    -
    claims_df["Date of Claim"]
).dt.days

In [18]:
claims_df["Approved Claim Flag"] = np.where(
    claims_df["Claim Status"]=="Approved",
    1,
    0
)

In [19]:
# Payment Intelligence

payment_df["Successful Payment Flag"] = np.where(
    payment_df["Payment Status"]=="Successful",
    1,
    0
)

In [20]:
payment_df["Failed Payment Flag"] = np.where(
    payment_df["Payment Status"]=="Failed",
    1,
    0
)

In [21]:
payment_df["Payment Month"] = (
    payment_df["Date of Payment"]
).dt.month_name()

In [22]:
payment_df["Payment Year"] = (
    payment_df["Date of Payment"]
).dt.year

In [23]:
# Risk Intelligence

def risk_bucket(score):

    if score <= 30:
        return "Low Risk"

    elif score <= 60:
        return "Medium Risk"

    else:
        return "High Risk"

In [24]:
additional_df["Risk Category"] = (
    additional_df["Risk Score"]
    .apply(risk_bucket)
)

In [25]:
additional_df["High Risk Flag"] = np.where(
    additional_df["Risk Score"] >= 70,
    1,
    0
)

In [26]:
def renewal_segment(status):

    if status=="Renewed":
        return "Loyal"

    elif status=="Pending":
        return "At Risk"

    else:
        return "Churned"

In [27]:
additional_df["Renewal Segment"] = (
    additional_df["Renewal Status"]
    .apply(renewal_segment)
)

In [28]:
insurance_master = customer_df.merge(
    policy_df,
    on="Customer ID",
    how="left"
)

In [29]:
insurance_master.shape

(6852, 22)

In [30]:
insurance_master = insurance_master.merge(
    additional_df,
    on="Policy ID",
    how="left"
)

In [31]:
insurance_master.head()

,Customer ID,Name,Gender,Age,Occupation,Marital Status,"Address (City, State, Zip Code)",Age Bucket,Married Flag,Senior Citizen Flag,Customer Segment,Policy ID,Policy Type,Coverage Amount,Premium Amount,Policy Start Date,Policy End Date,Payment Frequency,Status,Policy Duration Days,Policy Duration Category,Active Policy Flag,Agent ID,Renewal Status,Policy Discounts,Risk Score,High_Risk_Flag,Risk Category,High Risk Flag,Renewal Segment
0,CUST87425473,Norma Fisher,Female,63,Futures trader,Married,"Sullivanport, New Mexico, 47431",56+,1,1,Senior Adult,POL10637039,Property,285269.0,540.55,2018-07-20,2024-12-31,Annually,Terminated,2356.0,Long Term,0.0,AGT55023546,Pending,5.0,45.0,0.0,Medium Risk,0.0,At Risk
1,CUST52762998,Susan Wagner,Male,57,Herbalist,Single,"Port Cody, Kentucky, 70305",56+,0,0,Senior Adult,POL30067784,Health,402009.0,809.51,2015-01-04,2022-09-07,Annually,Lapsed,2803.0,Long Term,0.0,AGT12774099,Not Renewed,5.0,84.0,1.0,High Risk,1.0,Churned
2,CUST78641469,Dr. Stephanie Collins,Other,79,"Buyer, retail",Widowed,"Thomasville, Minnesota, 57408",56+,0,1,Senior Adult,NaN,NaN,NaN,NaN,NaT,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,CUST82850738,Martin Harris,Other,18,Sub,Widowed,"Ramoshaven, Kentucky, 08664",18-25,0,0,Young Adult,POL32316303,Auto,432298.0,1160.47,2020-12-24,2025-12-02,Annually,Terminated,1804.0,Long Term,0.0,AGT98177414,Renewed,16.0,2.0,0.0,Low Risk,0.0,Loyal
4,CUST83770928,Kimberly Smith,Male,75,Oceanographer,Single,"New Tristanmouth, Kansas, 96220",56+,0,1,Senior Adult,POL60672344,Auto,85463.0,961.64,2019-07-25,2029-06-05,Annually,Active,3603.0,Long Term,1.0,AGT74100066,Pending,28.0,91.0,1.0,High Risk,1.0,At Risk


In [32]:
insurance_master.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6852 entries, 0 to 6851
Data columns (total 30 columns):
 #   Column                           Non-Null Count  Dtype         
---  ------                           --------------  -----         
 0   Customer ID                      6852 non-null   object        
 1   Name                             6852 non-null   object        
 2   Gender                           6852 non-null   object        
 3   Age                              6852 non-null   int64         
 4   Occupation                       6852 non-null   object        
 5   Marital Status                   6852 non-null   object        
 6   Address (City, State, Zip Code)  6852 non-null   object        
 7   Age Bucket                       6852 non-null   object        
 8   Married Flag                     6852 non-null   int64         
 9   Senior Citizen Flag              6852 non-null   int64         
 10  Customer Segment                 6852 non-null   object     

In [33]:
insurance_master.isnull().sum()

Customer ID                           0
Name                                  0
Gender                                0
Age                                   0
Occupation                            0
Marital Status                        0
Address (City, State, Zip Code)       0
Age Bucket                            0
Married Flag                          0
Senior Citizen Flag                   0
Customer Segment                      0
Policy ID                          1852
Policy Type                        1852
Coverage Amount                    1852
Premium Amount                     1852
Policy Start Date                  1852
Policy End Date                    1852
Payment Frequency                  1852
Status                             1852
Policy Duration Days               1852
Policy Duration Category           1852
Active Policy Flag                 1852
Agent ID                           1852
Renewal Status                     1852
Policy Discounts                   1852


In [34]:
# Export

insurance_master.to_csv(
    "../data/processed/insurance_master_dataset.csv",
    index=False
)

In [35]:
claims_df.to_csv(
    "../data/processed/claims_fact.csv",
    index=False
)

In [36]:
payment_df.to_csv(
    "../data/processed/payment_fact.csv",
    index=False
)

In [37]:
print(insurance_master.shape)
print(claims_df.shape)
print(payment_df.shape)

(6852, 30)
(5000, 13)
(5000, 11)


In [38]:
customer_df["Customer ID"].duplicated().sum()

np.int64(0)

In [39]:
policy_df["Customer ID"].duplicated().sum()

np.int64(1852)

In [40]:
policy_df["Policy ID"].duplicated().sum()

np.int64(0)

In [41]:
additional_df["Policy ID"].duplicated().sum()

np.int64(0)

In [47]:
policy_df["Customer ID"].nunique()

3148

In [48]:
policy_df.shape

(5000, 12)

In [51]:
print(customer_df["Customer ID"].duplicated().sum())

print(policy_df["Customer ID"].duplicated().sum())

print(policy_df["Policy ID"].duplicated().sum())

print(additional_df["Policy ID"].duplicated().sum())

print(policy_df["Customer ID"].nunique())

0
1852
0
0
3148


In [52]:
policy_master = (
    policy_df
    .merge(
        customer_df,
        on="Customer ID",
        how="left"
    )
    .merge(
        additional_df,
        on="Policy ID",
        how="left"
    )
)

print(policy_master.shape)

(5000, 30)


In [53]:
policy_master.to_csv(
    "../data/processed/policy_master_dataset.csv",
    index=False
)

In [55]:
policy_master.shape

(5000, 30)